# Local Forecasting Demo

AI Assistance:
OpenAI ChatGPT was used for code debugging, code generation, code/repo organization,
and code methodological brainstorming. All final modeling, implementation,
validation, commentary, and interpretation were performed and verified by the authors.
Model used: GPT-5

**Real Databricks modeling-panel subset; local fitting and outputs.** This notebook reads a bounded sample of the same `gold_modeling_panel` used by the production forecast workflow. It uses actual point-in-time FHFA/ACS-derived features and targets, but fits locally on only 500 tracts. It does not read or write Databricks during normal execution after the subset has been downloaded.

The local tree model uses scikit-learn and mirrors the production script's currently selected one-year GBT settings; it is not an identical Spark ML implementation.

The sample is intentionally limited to the primary **one-year** horizon, eligible training rows, and observed validation rows. Calibration and locked temporal test data are excluded. Results are a small-subset demonstration—not national validation results, a replacement for the production scorecard, or a model approval.

The subset is cached under `reports/generated/meeting_forecast_demo/data/` (ignored by Git). The cache filename is versioned for the stable cross-vintage contract, so an older decade-filtered extract is not silently reused. If it is absent, the download cell can retrieve it with the Databricks CLI and a configured OAuth profile. Model results and charts are written under the same generated directory.

Install local dependencies from the repository root with `py -3.11 -m pip install -e ".[local-forecast]"`.

## 1. What the model predicts

For a tract and forecast origin year *t*, the target is the **cumulative log change** in the FHFA single-family house price index (HPI) from *t* to the end of the horizon:

```text
target = log(HPI at target year / HPI at origin year)
```

A positive target means the index rose; a negative target means it fell. This is not a prediction of an individual home's price, total housing value, rent, or neighborhood desirability. The notebook uses the same 26-feature registry as [`src/neighborhood_twin/forecasting.py`](../../src/neighborhood_twin/forecasting.py). The model inputs are:

- **Tract HPI history and trend:** `origin_annual_change_pct`, `hpi_change_1_year_pct`, `hpi_change_3_year_pct`, `hpi_change_5_year_pct`, `hpi_recent_3_year_mean_change_pct`, `hpi_recent_5_year_change_stddev_pct`, `hpi_history_observation_count`, `hpi_history_gap_count`
- **County and state trends:** `county_origin_mean_annual_change_pct`, `state_origin_mean_annual_change_pct`
- **Population and housing:** `population_total_estimate`, `housing_units_total_estimate`, `occupancy_rate_estimate`, `vacancy_rate_estimate`, `owner_occupancy_rate_estimate`, `renter_occupancy_rate_estimate`, `single_family_share_estimate`, `multifamily_share_estimate`, `mobile_home_share_estimate`
- **Income, value, rent, and data quality:** `median_household_income_estimate`, `median_owner_occupied_value_estimate`, `median_gross_rent_estimate`, `acs_measurement_age_years`, `median_household_income_relative_moe`, `median_owner_occupied_value_relative_moe`, `median_gross_rent_relative_moe`

The target is `target_cumulative_log_change`; it is the value being predicted, not one of the model inputs. The production workflow fits one model per horizon using only rows marked eligible for fitting. Median imputation is learned from training data; the linear model also standardizes features. Validation rows come from later, frozen origin years. This notebook follows those row filters for the one-year horizon.

In [ ]:
from __future__ import annotations

import csv
import math
import os
import subprocess
import sys
from collections import defaultdict
from pathlib import Path
from statistics import mean
from typing import Any

import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the project repository")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neighborhood_twin.forecasting import (  # noqa: E402
    GBT_MAX_ITER,
    GBT_SEED,
    GBT_STEP_SIZE,
    GBT_SUBSAMPLING_RATE,
    LINEAR_FEATURE_COLUMNS,
    annualized_rate_from_cumulative_log,
    cumulative_log_from_annual_rate,
)

HORIZON = 1
GBT_DEMO_MAX_DEPTH = 2
TRACT_SAMPLE_SIZE = 500
DATA_DIR = PROJECT_ROOT / "reports" / "generated" / "meeting_forecast_demo" / "data"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "generated" / "meeting_forecast_demo"
PANEL_CSV = DATA_DIR / "gold_modeling_panel_1yr_stable_cross_vintage_500_tracts.csv"
DATABRICKS_PROFILE = os.environ.get("DATABRICKS_PROFILE") or os.environ.get(
    "DATABRICKS_CONFIG_PROFILE"
)
REFRESH_DATA = False
print(f"Forecast feature registry: {len(LINEAR_FEATURE_COLUMNS)} inputs")
print(f"Local panel cache: {PANEL_CSV}")

## 2. Load a bounded subset of the production panel

The source is `workspace.neighborhood_twin_gold.gold_modeling_panel`, the same governed panel read by the Databricks forecasting workflow. The query deterministically chooses 500 tracts using a stable hash (rather than taking the first 500 GEOIDs, which would overrepresent a small geographic area). It includes only one-year rows that are either eligible training examples with observed targets or observed validation examples. Eligibility follows the production tract-vintage contract; target windows are not removed merely because they cross a calendar-decade boundary.

In [ ]:
def download_panel_subset(profile: str) -> None:
    """Download the reproducible teaching subset, never the full panel."""
    if not profile:
        raise RuntimeError(
            "Set DATABRICKS_PROFILE to a configured OAuth profile before downloading, "
            "or use the already cached local CSV."
        )
    columns = [
        "geoid_2020",
        "state_fips",
        "state_abbreviation",
        "state_name",
        "county_geoid",
        "county_name",
        "region_name",
        "forecast_origin_year",
        "horizon_years",
        "target_end_year",
        "origin_hpi",
        "target_cumulative_log_change",
        "target_annualized_change_pct",
        "target_is_observed",
        "temporal_role",
        "geographic_role",
        "eligible_for_point_model_fit",
        "eligible_for_locked_temporal_evaluation",
        "eligible_for_geographic_evaluation",
        "modeling_cohort",
        "model_readiness_build_id",
        "model_readiness_built_at_utc",
        *LINEAR_FEATURE_COLUMNS,
    ]
    select_list = ",\n  ".join(f"p.`{column}`" for column in columns)
    sql = f"""WITH sampled_tracts AS (
  SELECT DISTINCT geoid_2020
  FROM workspace.neighborhood_twin_gold.gold_modeling_panel
  WHERE horizon_years = {HORIZON}
    AND target_is_observed = TRUE
    AND (eligible_for_point_model_fit = TRUE OR temporal_role = 'validation')
  ORDER BY xxhash64(geoid_2020), geoid_2020
  LIMIT {TRACT_SAMPLE_SIZE}
)
SELECT
  {select_list}
FROM workspace.neighborhood_twin_gold.gold_modeling_panel p
INNER JOIN sampled_tracts s ON p.geoid_2020 = s.geoid_2020
WHERE p.horizon_years = {HORIZON}
  AND p.target_is_observed = TRUE
  AND (p.eligible_for_point_model_fit = TRUE OR p.temporal_role = 'validation')
ORDER BY p.geoid_2020, p.forecast_origin_year"""
    command = [
        "databricks",
        "experimental",
        "aitools",
        "tools",
        "query",
        "--profile",
        profile,
        "--output",
        "csv",
        sql,
    ]
    result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", timeout=600)
    if result.returncode != 0:
        raise RuntimeError(f"Databricks subset query failed: {result.stderr.strip()}")
    if not result.stdout.strip():
        raise RuntimeError(f"Databricks returned no CSV rows. CLI output: {result.stderr}")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    PANEL_CSV.write_text(result.stdout, encoding="utf-8-sig", newline="")
    print(f"Downloaded bounded panel subset to {PANEL_CSV}")


if REFRESH_DATA or not PANEL_CSV.is_file():
    download_panel_subset(DATABRICKS_PROFILE)
else:
    print("Using cached local panel subset; set REFRESH_DATA = True to refresh it.")

## 3. Check the sample and frozen split

The model-fitting flag—not a random row split—determines which observations can train the model. Validation is kept separate and is used only to score predictions. Repeated observations from the same tract across years are expected, the split is temporal.

In [ ]:
numeric_fields = {
    *LINEAR_FEATURE_COLUMNS,
    "forecast_origin_year",
    "horizon_years",
    "target_end_year",
    "origin_hpi",
    "target_cumulative_log_change",
    "target_annualized_change_pct",
}
boolean_fields = {
    "target_is_observed",
    "eligible_for_point_model_fit",
    "eligible_for_locked_temporal_evaluation",
    "eligible_for_geographic_evaluation",
}
with PANEL_CSV.open(encoding="utf-8-sig", newline="") as handle:
    raw_rows = list(csv.DictReader(handle))
if not raw_rows:
    raise ValueError(f"The local panel subset is empty: {PANEL_CSV}")
missing_features = set(LINEAR_FEATURE_COLUMNS) - set(raw_rows[0])
if missing_features:
    raise ValueError(f"Panel export is missing registered features: {sorted(missing_features)}")

panel = []
for raw in raw_rows:
    row: dict[str, Any] = dict(raw)
    for name in numeric_fields:
        value = row.get(name)
        row[name] = float(value) if value not in (None, "") else None
    for name in boolean_fields:
        row[name] = str(row.get(name, "")).strip().lower() == "true"
    panel.append(row)

training_rows = [
    row
    for row in panel
    if row["temporal_role"] == "training"
    and row["eligible_for_point_model_fit"]
    and row["target_is_observed"]
]
validation_rows = [
    row for row in panel if row["temporal_role"] == "validation" and row["target_is_observed"]
]
if not training_rows or not validation_rows:
    raise ValueError("The local panel subset must contain eligible training and validation rows")
if any(row["horizon_years"] != HORIZON for row in panel):
    raise ValueError("The teaching subset contains an unexpected forecast horizon")
if {row["temporal_role"] for row in panel} - {"training", "validation"}:
    raise ValueError("The local panel unexpectedly includes a reserved or ineligible role")
if any(row["eligible_for_locked_temporal_evaluation"] for row in panel):
    raise ValueError("The local teaching subset must not include locked-test rows")

tract_count = len({row["geoid_2020"] for row in panel})
build_ids = sorted({row["model_readiness_build_id"] for row in panel})
training_origin_count = len({row["forecast_origin_year"] for row in training_rows})
validation_origin_count = len({row["forecast_origin_year"] for row in validation_rows})
print(f"Rows: {len(panel):,}; distinct tracts: {tract_count:,}; horizon: {HORIZON} year")
print(f"Training: {len(training_rows):,} rows across {training_origin_count} origins")
print(f"Validation: {len(validation_rows):,} rows across {validation_origin_count} origins")
print(f"Training origins: {sorted({int(r['forecast_origin_year']) for r in training_rows})}")
print(f"Validation origins: {sorted({int(r['forecast_origin_year']) for r in validation_rows})}")
print(f"Model-readiness build ID(s): {build_ids}")

## 4. Fit local candidates and compare with simple rules

The local code uses the same registered features, target, horizon, and training/validation rows as the production workflow. It fits a regularized linear model and a gradient-boosted-tree model, then scores the five fixed baselines: zero change, latest-change persistence, recent three-year mean, county trend, and state trend.

The production script uses Spark ML `GBTRegressor` and currently selects depth 2 with 50 iterations for the one-year horizon via purged, training-only temporal cross-validation. To keep this bounded demo runnable locally, it uses scikit-learn `GradientBoostingRegressor` with matching one-year settings (depth 2, 50 estimators, learning rate 0.05, subsample 0.8, seed 699). The implementations can differ slightly, so local scores are not expected to match the production scorecard exactly; this demo pins the current selected settings rather than re-tuning the 500-tract sample.

In [ ]:
BASELINE_FEATURES = {
    "latest_change_persistence": "hpi_change_1_year_pct",
    "recent_three_year_mean": "hpi_recent_3_year_mean_change_pct",
    "county_current_trend": "county_origin_mean_annual_change_pct",
    "state_current_trend": "state_origin_mean_annual_change_pct",
}
x_train = [[row[name] for name in LINEAR_FEATURE_COLUMNS] for row in training_rows]
y_train = [row["target_cumulative_log_change"] for row in training_rows]
x_validation = [[row[name] for name in LINEAR_FEATURE_COLUMNS] for row in validation_rows]

local_models = {
    "regularized_linear": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=0.05),
    ),
    "gradient_boosted_trees": make_pipeline(
        SimpleImputer(strategy="median"),
        GradientBoostingRegressor(
            n_estimators=GBT_MAX_ITER,
            learning_rate=GBT_STEP_SIZE,
            max_depth=GBT_DEMO_MAX_DEPTH,
            loss="squared_error",
            subsample=GBT_SUBSAMPLING_RATE,
            random_state=GBT_SEED,
        ),
    ),
}
predictions: list[dict[str, Any]] = []


def add_predictions(model_name: str, values: list[float], rows: list[dict[str, Any]]) -> None:
    for row, prediction in zip(rows, values, strict=True):
        actual = row["target_cumulative_log_change"]
        predictions.append(
            {
                "geoid_2020": row["geoid_2020"],
                "state_name": row["state_name"],
                "county_name": row["county_name"],
                "forecast_origin_year": int(row["forecast_origin_year"]),
                "horizon_years": HORIZON,
                "model_name": model_name,
                "target_cumulative_log_change": actual,
                "prediction_cumulative_log_change": float(prediction),
                "residual_target_minus_prediction": actual - float(prediction),
                "target_annualized_change_pct": annualized_rate_from_cumulative_log(
                    actual, HORIZON
                ),
                "prediction_annualized_change_pct": annualized_rate_from_cumulative_log(
                    float(prediction), HORIZON
                ),
            }
        )


for name, estimator in local_models.items():
    estimator.fit(x_train, y_train)
    add_predictions(name, estimator.predict(x_validation).tolist(), validation_rows)

for row in validation_rows:
    add_predictions("zero_change", [0.0], [row])
    for model_name, feature in BASELINE_FEATURES.items():
        annual_rate = row[feature]
        if annual_rate is not None and annual_rate > -100:
            add_predictions(
                model_name,
                [cumulative_log_from_annual_rate(annual_rate, HORIZON)],
                [row],
            )

grouped_predictions: dict[str, list[dict[str, Any]]] = defaultdict(list)
for row in predictions:
    grouped_predictions[row["model_name"]].append(row)
metrics = []
for model_name, rows in sorted(grouped_predictions.items()):
    errors = [row["residual_target_minus_prediction"] for row in rows]
    annualized_errors = [
        row["target_annualized_change_pct"] - row["prediction_annualized_change_pct"]
        for row in rows
    ]
    target_mean = mean(row["target_cumulative_log_change"] for row in rows)
    total_sum_squares = sum(
        (row["target_cumulative_log_change"] - target_mean) ** 2 for row in rows
    )
    residual_sum_squares = sum(error**2 for error in errors)
    metrics.append(
        {
            "model_name": model_name,
            "observation_count": len(rows),
            "mae_log_change": mean(abs(error) for error in errors),
            "rmse_log_change": math.sqrt(mean(error**2 for error in errors)),
            "mean_signed_error_target_minus_prediction": mean(errors),
            "mae_annualized_percentage_points": mean(abs(error) for error in annualized_errors),
            "r_squared": 1 - residual_sum_squares / total_sum_squares
            if total_sum_squares
            else None,
        }
    )
zero_mae = next(row["mae_log_change"] for row in metrics if row["model_name"] == "zero_change")
simple_scores = [row for row in metrics if row["model_name"] in BASELINE_FEATURES]
strongest_simple = min(simple_scores, key=lambda row: row["mae_log_change"])
for row in metrics:
    row["mae_skill_vs_zero_change"] = 1 - row["mae_log_change"] / zero_mae if zero_mae else None
    row["strongest_simple_baseline"] = strongest_simple["model_name"]
    row["mae_improvement_vs_strongest_simple"] = (
        1 - row["mae_log_change"] / strongest_simple["mae_log_change"]
        if strongest_simple["mae_log_change"]
        else None
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with (OUTPUT_DIR / "local_subset_predictions.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(handle, fieldnames=list(predictions[0]))
    writer.writeheader()
    writer.writerows(predictions)
with (OUTPUT_DIR / "local_subset_metrics.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(metrics[0]))
    writer.writeheader()
    writer.writerows(metrics)

print(f"Training rows: {len(training_rows):,}; validation rows: {len(validation_rows):,}")
print(f"Strongest simple baseline by validation MAE: {strongest_simple['model_name']}")
print(f"Saved metrics and per-tract predictions under {OUTPUT_DIR}")

## 5. One-year validation accuracy

- **MAE (log change)** is the average absolute gap between the predicted and actual one-year cumulative log HPI change across validation rows. Lower means closer predictions. The values are in cumulative log-change units, not dollars or percentage points.
- **RMSE (log change)** also measures average error, but penalizes a few large misses more heavily than MAE.
- **Mean signed error = actual minus prediction.** A positive value means the predictions tend to be too low; a negative value means they tend to be too high.
- **Annualized MAE** expresses the difference between actual and predicted annualized rates in percentage points.
- **MAE skill vs zero change** is `1 - model MAE / zero-change MAE`. Positive means better than predicting no change; negative means worse.

The next cell displays two grouped-bar charts, both restricted to the one-year horizon: one from the local 500-tract validation sample and one from the latest full-data production scorecard available in `reports/generated/forecast_error_analysis/`. Bars show cumulative log-change MAE; lower is better. The local and production samples are different, so compare methods within each chart rather than treating the two charts as a paired experiment. Some methods have slightly different observation counts when required baseline inputs are missing. Five-year results are excluded.


In [ ]:
figure_dir = OUTPUT_DIR / "local_subset_figures"
figure_dir.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = [
    "zero_change",
    "latest_change_persistence",
    "recent_three_year_mean",
    "county_current_trend",
    "state_current_trend",
    "regularized_linear",
    "gradient_boosted_trees",
]
MODEL_LABELS = {
    "zero_change": "Zero change",
    "latest_change_persistence": "Latest-change persistence",
    "recent_three_year_mean": "Recent 3-year mean",
    "county_current_trend": "County trend",
    "state_current_trend": "State trend",
    "regularized_linear": "Regularized linear",
    "gradient_boosted_trees": "Gradient-boosted trees",
}
MODEL_COLORS = {
    "zero_change": "#9E9E9E",
    "latest_change_persistence": "#8D6E63",
    "recent_three_year_mean": "#FFB300",
    "county_current_trend": "#5E35B1",
    "state_current_trend": "#00897B",
    "regularized_linear": "#1565C0",
    "gradient_boosted_trees": "#D81B60",
}


def plot_one_year_mae(
    rows: list[dict[str, Any]],
    *,
    title: str,
    note: str,
    filename: str,
) -> None:
    rows_by_model: dict[str, dict[str, Any]] = {}
    for row in rows:
        model_name = str(row["model_name"])
        if model_name in rows_by_model:
            raise ValueError(f"Duplicate one-year scorecard row for {model_name}")
        rows_by_model[model_name] = row
    if set(rows_by_model) != set(MODEL_ORDER):
        raise ValueError(
            f"Expected one scorecard row for each of {MODEL_ORDER}; found {sorted(rows_by_model)}"
        )

    figure, axis = plt.subplots(figsize=(11, 6))
    bar_width = 0.82 / len(MODEL_ORDER)
    for index, model_name in enumerate(MODEL_ORDER):
        error = float(rows_by_model[model_name]["mae_log_change"])
        if not math.isfinite(error) or error < 0:
            raise ValueError(f"Invalid MAE for {model_name}: {error}")
        offset = (index - (len(MODEL_ORDER) - 1) / 2) * bar_width
        axis.bar(
            offset,
            error,
            width=bar_width,
            label=MODEL_LABELS[model_name],
            color=MODEL_COLORS[model_name],
        )
    axis.set_xticks([0], ["1-year"])
    axis.set_xlabel("Forecast horizon")
    axis.set_ylabel("Mean absolute cumulative log-change error")
    figure.suptitle(title, y=0.97)
    axis.grid(axis="y", color="#CFD8DC", linewidth=0.7, alpha=0.7)
    axis.set_axisbelow(True)
    handles, labels = axis.get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        ncol=4,
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.91),
    )
    figure.subplots_adjust(bottom=0.19, top=0.68)
    figure.text(0.01, 0.01, note, ha="left", va="bottom", fontsize=8.5, color="#607D8B")
    figure.savefig(figure_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def plot_all_horizons_mae(
    rows: list[dict[str, Any]],
    *,
    title: str,
    note: str,
    filename: str,
) -> None:
    rows_by_horizon: dict[int, dict[str, dict[str, Any]]] = {}
    for row in rows:
        horizon = int(row["horizon_years"])
        model_name = str(row["model_name"])
        horizon_rows = rows_by_horizon.setdefault(horizon, {})
        if model_name in horizon_rows:
            raise ValueError(f"Duplicate {horizon}-year scorecard row for {model_name}")
        horizon_rows[model_name] = row

    horizons = sorted(rows_by_horizon)
    if not horizons:
        raise ValueError("Production scorecard has no forecast horizons")
    for horizon, horizon_rows in rows_by_horizon.items():
        if set(horizon_rows) != set(MODEL_ORDER):
            raise ValueError(
                f"Expected one row for each model at {horizon} years; found {sorted(horizon_rows)}"
            )

    figure, axis = plt.subplots(figsize=(13, 6))
    bar_width = 0.82 / len(MODEL_ORDER)
    for index, model_name in enumerate(MODEL_ORDER):
        values = []
        for horizon in horizons:
            error = float(rows_by_horizon[horizon][model_name]["mae_log_change"])
            if not math.isfinite(error) or error < 0:
                raise ValueError(f"Invalid {horizon}-year MAE for {model_name}: {error}")
            values.append(error)
        offset = (index - (len(MODEL_ORDER) - 1) / 2) * bar_width
        axis.bar(
            [position + offset for position in range(len(horizons))],
            values,
            width=bar_width,
            label=MODEL_LABELS[model_name],
            color=MODEL_COLORS[model_name],
        )
    axis.set_xticks(range(len(horizons)), [f"{horizon}-year" for horizon in horizons])
    axis.set_xlabel("Forecast horizon")
    axis.set_ylabel("Mean absolute cumulative log-change error")
    figure.suptitle(title, y=0.97)
    axis.grid(axis="y", color="#CFD8DC", linewidth=0.7, alpha=0.7)
    axis.set_axisbelow(True)
    handles, labels = axis.get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        ncol=4,
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.91),
    )
    figure.subplots_adjust(bottom=0.19, top=0.68)
    figure.text(0.01, 0.01, note, ha="left", va="bottom", fontsize=8.5, color="#607D8B")
    figure.savefig(figure_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(figure)


scorecard_dir = PROJECT_ROOT / "reports" / "generated" / "forecast_error_analysis"
scorecard_files = sorted(scorecard_dir.glob("full_data_validation_scorecard_*.csv"))
if not scorecard_files:
    raise FileNotFoundError(
        f"No full-data validation scorecard found under {scorecard_dir}; "
        "export the latest production scorecard before running this cell."
    )
production_scorecard_path = scorecard_files[-1]
with production_scorecard_path.open(encoding="utf-8-sig", newline="") as handle:
    production_scorecard = list(csv.DictReader(handle))
production_rows = production_scorecard
if not production_rows:
    raise ValueError(f"No production rows found in {production_scorecard_path}")
production_horizons = sorted({int(row["horizon_years"]) for row in production_rows})
production_run_ids = {row["forecast_run_id"] for row in production_rows}
production_built_at = {row["forecast_built_at_utc"] for row in production_rows}
if len(production_run_ids) != 1 or len(production_built_at) != 1:
    raise ValueError(f"Production scorecard mixes runs or build times: {production_scorecard_path}")
production_run_id = next(iter(production_run_ids))
production_run_built_at = next(iter(production_built_at))

plot_all_horizons_mae(
    production_rows,
    title="Production model validation MAE by method - all horizons",
    note=(
        f"Production run {production_run_id}, built {production_run_built_at}; "
        f"horizons: {', '.join(f'{horizon}-year' for horizon in production_horizons)}. "
        "Validation-only; the 5-year result is exploratory given limited independent history. "
        "Lower is better."
    ),
    filename="validation_mae_production_all_horizons.png",
)
plot_one_year_mae(
    metrics,
    title="Local subset validation MAE by method - one year",
    note=(
        "Local 500-tract validation sample. Method sample sizes vary slightly where baseline "
        "inputs are missing. Lower is better."
    ),
    filename="validation_mae_local_subset.png",
)

ordered_metrics = sorted(metrics, key=lambda row: row["mae_log_change"])
print(f"Latest production source: {production_scorecard_path.name}")
print(f"Production run: {production_run_id} ({production_run_built_at})")
print(f"{'Model':<30} {'N':>6} {'MAE':>10} {'RMSE':>10} {'Bias':>10} {'Skill vs zero':>14}")
for row in ordered_metrics:
    print(
        f"{row['model_name']:<30} {row['observation_count']:>6} "
        f"{row['mae_log_change']:>10.5f} {row['rmse_log_change']:>10.5f} "
        f"{row['mean_signed_error_target_minus_prediction']:>10.5f} "
        f"{row['mae_skill_vs_zero_change']:>13.1%}"
    )